# Recuperar artefactos primarios de Drive al repo

Notebook utilitario, reentrante, sin entrenar nada. Su objetivo es **cerrar la asimetria de trazabilidad** entre el SSL central y el resto del proyecto:

- Hasta ahora, los `run_info.json`, `posthoc_analysis.*` y `config.yaml` del SSL central (smoke / coverage / pilot / full) vivian **solo en Drive**. El repo solo tenia `results/pretraining/ssl_sampling_plan.csv`. Eso hace que la convergencia central no sea auditable sin acceso a Drive.
- En cambio, el bloque federado (`results/pretraining_federated/`) y todo el downstream ya tenian `run_info.json` versionados.

Este notebook trae al repo los artefactos primarios (JSONs, MDs, configs, sampling_plan, label_mappings, metrics.jsonl ligeros y logs textuales pequenos). NO trae:

- `metrics.jsonl` del SSL central (decenas/cientos de MB).
- Checkpoints `.pt` (~9.7 MB cada uno).
- Shards `processed/`, `processed_downstream/`, `raw/`.
- `audit/<DATASET>.json` por dataset (ya consolidado en `audit_report.{csv,md}` + `audit_summary.json`).

Bloques que copia:

| Bloque | Que cubre |
|---|---|
| A | SSL central (4 stages) + posthoc del full + README generado |
| B | SSL federated (dry_run / config / metrics_round_summary que faltaban) |
| C | Downstream classification `metrics.jsonl` por (dataset, modo) |
| D | Downstream CMAPSS RUL `config.yaml` + `metrics.jsonl` |
| E | Downstream federated configs + label_mappings + metrics CWRU/HSG18 |
| F | Logs textuales historicos (audit, harmonization, write_shards) con cap 5 MB |

Salida: `results/pretraining/`, `results/pretraining_federated/`, `results/downstream/`, `results/audit/`, `results/processed_run_logs/` quedan completos. **No se hace commit ni push automatico**: al final se imprimen los comandos sugeridos.

## 1. Setup minimo de Colab

Mount de Drive + cd al repo. El script luego asume que la cwd es la raiz del repo (donde esta `CLAUDE.md`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!bash /content/drive/MyDrive/fm_fl_phmd/colab_init.sh
%cd /content/fm_fl_phmd

## 2. Verificacion de estado limpio del repo

Recomendado: empezar con working tree limpio para que el `git status` final muestre claramente lo que el script copio.

In [ ]:
!git status --short
!git log --oneline -3

## 3. Dry-run: ver el plan sin copiar nada

Primera pasada para confirmar que todas las rutas Drive existen y cuanto se va a copiar. Si algun fichero falta en Drive, aparece como `missing_src` en la tabla resumen.

In [ ]:
!python notebooks/utils/recover_drive_artifacts.py --dry-run

## 4. Ejecucion real

Si el dry-run reporta lo esperado (los unicos `missing_src` aceptables son los del Bloque B que dependen de si esos ficheros existen en Drive), lanzamos la copia real.

El script:

1. Copia los ~65 ficheros listados, omitiendo los que ya estan identicos por SHA-256.
2. Cubre el Bloque F (logs textuales) con un cap de 5 MB por defecto. Si alguno excede, queda como `too_large`.
3. Genera `results/pretraining/README.md` con tabla resumen + bloque por stage, leyendo dinamicamente los `run_info.json` recien copiados.
4. Aplica sanity checks duros sobre los `run_info.json` del SSL central: `coverage_pass`/`pilot_pass`/`smoke_pass` esperado, `config_hash` esperado, `param_count` esperado.
5. Si algun sanity falla, el script sale con codigo 2 y la tabla muestra el desajuste exacto. Nada queda comprometido (los ficheros se quedan donde estaban, pero el commit deberia inspeccionarse antes).

In [ ]:
!python notebooks/utils/recover_drive_artifacts.py

## 5. Revision del diff antes de commitear

Inspeccion rapida del diff. Lo esperable:

- ~15 ficheros nuevos bajo `results/pretraining/` (las 4 stages + README + posthoc).
- Ficheros nuevos bajo `results/downstream/{cwru,hsg18,pbcp16,phm18,cmapss_rul}/<mode>/metrics.jsonl` (curvas por epoch).
- Ficheros nuevos bajo `results/downstream/fl_*_vs_central/<ds>/<mode>/{config.yaml, label_mapping.json, metrics.jsonl}`.
- Opcionalmente: `results/audit/run_log.log`, `results/processed_run_logs/full_20260521T141247Z.log`, `results/downstream/cmapss_rul_decision/write_shards.log`.

Si algo no cuadra, parar aqui y revisar.

In [ ]:
!git status
print('---')
!git diff --stat results/ | tail -30

## 6. Sanity de los run_info copiados (cross-check con CLAUDE.md sec 17)

Lectura rapida de los 4 `run_info.json` centrales recien commiteados para confirmar que las metricas internas matchean los valores que CLAUDE.md sec 17 cita textualmente.

In [ ]:
import json
from pathlib import Path

stages = [
    ('ssl_central_smoke',    'smoke_pass',    '46628aedb05becd6'),
    ('ssl_central_coverage', 'coverage_pass', 'e5cfd3b0684c7918'),
    ('ssl_central_pilot',    'pilot_pass',    'e4970c173c9dc244'),
    ('ssl_central_full',     'coverage_pass', '9ed84508a6820265'),
]
for stage, pass_key, expected_hash in stages:
    path = Path(f'results/pretraining/{stage}/run_info.json')
    if not path.is_file():
        print(f'  [MISS] {stage}: no se copio')
        continue
    info = json.loads(path.read_text(encoding='utf-8'))
    ok_pass = bool(info.get(pass_key))
    ok_hash = info.get('config_hash') == expected_hash
    flag = 'OK' if (ok_pass and ok_hash) else 'BAD'
    print(
        f'  [{flag}] {stage:24s}  '
        f'{pass_key}={info.get(pass_key)} '
        f'hash={info.get("config_hash")} '
        f'params={info.get("param_count")} '
        f'opt_steps={info.get("optimizer_steps", info.get("optimizer_steps_total"))}'
    )

## 7. Commit sugerido (manual)

**El notebook NO commitea ni hace push automatico**. Si quieres dejar la sesion en estado limpio, ejecuta una de las dos siguientes celdas a mano segun el caso.

Mensaje sugerido y agrupado por area (mas legible para el log):

In [ ]:
# Opcion 1: commit todo en uno (lo mas directo).
# Descomenta para ejecutar.
#
# !git add results/pretraining/ \
#          results/pretraining_federated/ \
#          results/downstream/ \
#          results/audit/ \
#          results/processed_run_logs/
# !git commit -m "chore(results): traer artefactos primarios SSL central y completar simetria con federado/downstream"
# !git status

In [ ]:
# Opcion 2: commits separados por area (mas granular en git log).
# Descomenta para ejecutar.
#
# !git add results/pretraining/
# !git commit -m "chore(results/pretraining): traer run_info, posthoc, configs y sampling_plans del SSL central desde Drive"
#
# !git add results/pretraining_federated/
# !git commit -m "chore(results/pretraining_federated): completar configs y dry_run_report/metrics_round_summary que faltaban"
#
# !git add results/downstream/
# !git commit -m "chore(results/downstream): traer metrics.jsonl, configs y label_mappings que faltaban (classification + RUL + FL-vs-central)"
#
# !git add results/audit/ results/processed_run_logs/
# !git commit -m "chore(results/audit): traer logs textuales del audit, harmonization y write_shards desde Drive (cap 5MB)"
#
# !git status
# !git log --oneline -10

## 8. Push (manual, solo si tu lo decides)

El push a `main` queda explicitamente fuera del flujo automatico. Si quieres subirlo, ejecuta:

```bash
!git push origin main
```

Si prefieres pasar por PR (lo recomendable para cambios grandes en `results/`):

```bash
!git checkout -b chore/results-trazabilidad-central
!git push -u origin chore/results-trazabilidad-central
# luego en GitHub: abrir PR a main
```